In [ ]:
import pandas as pd
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

### Because Noah-MP Mosaic and HUE requires 50 landcover catagories, ahead of running ungrib, you MUST run this script
- this script simply opens up the geogrid file that uses the NLCD 40 landcovers
- extracts the land use fractions
- creates a 50 landcover cube
- repalces the first 40 zeros with NLCD data
- and then replaces landusef in the geogrid files
- Finally, it changes the meta-data that is read by metgrid to ensure that it will not fail out

In [11]:
## Current iteration of the land-cover are enumerated from the top left corner, while WRF indexed from the bottom left corner. 
## we need to fix this so that we are able to run WRF

## load in Geogrid data
geogrid_data = xr.open_dataset('./master-geogrid/geo_em.d02.nc',engine='netcdf4')

<xarray.DataArray 'XLAT_M' (Time: 1, south_north: 120, west_east: 150)> Size: 72kB
[18000 values with dtype=float32]
Dimensions without coordinates: Time, south_north, west_east
Attributes:
    FieldType:    104
    MemoryOrder:  XY 
    units:        degrees latitude
    description:  Latitude on mass grid
    stagger:      M
    sr_x:         1
    sr_y:         1

In [13]:
## grab the land-use from the wrf data, and load into a numpy array
landuse_f = geogrid_data.LANDUSEF
landusef_numpy = np.zeros((1,50,120,150))
landusef_numpy[0,0:40,:,:] = landuse_f.values

In [ ]:
## just incase there is a perenial glacier in the mix, this must be removed
landusef_numpy[0,14,:,:] = 0.
landusef_numpy[0,21,:,:] = 0.

In [14]:
LU_dataarray = xr.DataArray(
    landusef_numpy,
#     coords={
#         "Time": geogrid_data.dims['Time'],
#         "south_north": geogrid_data.dims['south_north'],
#         "west_east": geogrid_data.dims['west_east'],
#         "land_cat": 47,
#     },
    dims=["Time", "land_cat","south_north","west_east"],
)

In [15]:
geogrid_data = geogrid_data.drop_vars('LANDUSEF')
geogrid_data.update({"LANDUSEF":LU_dataarray})

<xarray.Dataset> Size: 16MB
Dimensions:     (Time: 1, south_north: 120, west_east: 150,
                 south_north_stag: 121, west_east_stag: 151, soil_cat: 16,
                 month: 12, dust_erosion_dimension: 3, land_cat: 50)
Dimensions without coordinates: Time, south_north, west_east, south_north_stag,
                                west_east_stag, soil_cat, month,
                                dust_erosion_dimension, land_cat
Data variables: (12/60)
    Times       (Time) |S19 19B ...
    XLAT_M      (Time, south_north, west_east) float32 72kB ...
    XLONG_M     (Time, south_north, west_east) float32 72kB ...
    XLAT_V      (Time, south_north_stag, west_east) float32 73kB ...
    XLONG_V     (Time, south_north_stag, west_east) float32 73kB ...
    XLAT_U      (Time, south_north, west_east_stag) float32 72kB ...
    ...          ...
    CANFRA      (Time, south_north, west_east) float32 72kB ...
    EROD        (Time, dust_erosion_dimension, south_north, west_east) float32 216kB ...
    CLAYFRAC    (Time, south_north, west_east) float32 72kB ...
    SANDFRAC    (Time, south_north, west_east) float32 72kB ...
    IRRIGATION  (Time, south_north, west_east) float32 72kB ...
    LANDUSEF    (Time, land_cat, south_north, west_east) float64 7MB 0.0 ... 0.0
Attributes: (12/54)
    TITLE:                           OUTPUT FROM GEOGRID V4.2
    SIMULATION_START_DATE:           0000-00-00_00:00:00
    WEST-EAST_GRID_DIMENSION:        151
    SOUTH-NORTH_GRID_DIMENSION:      121
    BOTTOM-TOP_GRID_DIMENSION:       0
    WEST-EAST_PATCH_START_UNSTAG:    1
    ...                              ...
    FLAG_FRC_URB2D:                  1
    FLAG_IMPERV:                     1
    FLAG_CANFRA:                     1
    FLAG_EROD:                       1
    FLAG_CLAYFRAC:                   1
    FLAG_SANDFRAC:                   1

In [16]:
geogrid_data.attrs['NUM_LAND_CAT'] = int(50)
geogrid_data.attrs['NUM_LAND_CAT'] = np.int32(50)
geogrid_data['LANDUSEF'] = geogrid_data.LANDUSEF.astype(np.float32)
geogrid_data['LANDUSEF'].attrs['FieldType'] = np.int32(104)
geogrid_data['LANDUSEF'].attrs['MemoryOrder'] = 'XYZ'
geogrid_data['LANDUSEF'].attrs['units'] = 'category'
geogrid_data['LANDUSEF'].attrs['description'] = '2011 NLCD and MODIS land cover fraction data'
geogrid_data['LANDUSEF'].attrs['stagger']= 'M'
geogrid_data['LANDUSEF'].attrs['sr_x']=np.int32(1)
geogrid_data['LANDUSEF'].attrs['sr_y']=np.int32(1)

In [17]:
geogrid_data.to_netcdf('./geo_em.d02.50Catagories.nc')

## test data